<div dir="rtl" style="direction:rtl;text-align:right;font:750 20px Tahoma,Arial,sans-serif;line-height:2">
# KhaledDSML 0.9 · مرصد جودة البيانات المفقودة

## Little MCAR · تحليل ارتباط الفقد · الاستكمال المتعدد · Rubin

تطبيق بحثي بالعربية والإنجليزية ضمن **Analytics Studio** بتدرجات Aurora وOcean وAmethyst، وبطاقات إحصائية واضحة وجداول عناوينها العربية يمينًا والإنجليزية يسارًا.

**نسخة التطوير 0.9.0** غير منشورة؛ الإصدار الرسمي الحالي 0.8.0. جميع البيانات اصطناعية للأغراض التعليمية.
</div>

<div dir="rtl" style="text-align:right">
## 1. تجهيز المكتبة

نفّذ أوامر التثبيت، ثم اختر **Runtime → Restart session** قبل بقية الخلايا.
</div>

In [ ]:
%pip install --upgrade "khaled-dsml[econometrics]==0.8.0"
%pip install --no-deps --no-cache-dir --force-reinstall "git+https://github.com/noble2013/KhaledDSML.git@feature/v0.9-missing-data-inference"

<div dir="rtl" style="text-align:right">
## 2. إنشاء بيانات اقتصادية تعليمية

سنولد ثلاثة مؤشرات مترابطة، ثم نُدخل فقدًا اصطناعيًا في بعض المشاهدات. لا نغير القيم الموجودة.
</div>

In [ ]:
import numpy as np
import pandas as pd
import khaled_dsml as kd
from IPython.display import display

print("KhaledDSML version:",kd.__version__)
rng=np.random.default_rng(2026)
dataset=pd.DataFrame(
    rng.multivariate_normal(
        [450,150,85],[[1800,120,80],[120,240,20],[80,20,90]],
        size=220
    ),
    columns=["الدخل","الإنفاق","الرضا"]
)
dataset.loc[::7,"الإنفاق"]=np.nan
dataset.loc[::11,"الرضا"]=np.nan
display(dataset.head())

<div dir="rtl" style="text-align:right">
## 3. فحص فرضية MCAR باختبار Little

عدم رفض الفرضية الصفرية لا يثبت الفقد العشوائي التام، ورفضها لا يحدد وحده MAR أو MNAR. تختبر الدالة فرضيات تقاربية ونموذجًا اعتداليًا متعدد المتغيرات.
</div>

In [ ]:
little=kd.little_mcar_test(dataset)
kd.display_analysis_dashboard(
    little,theme="aurora",
    title_ar="تقرير Little لآلية فقد البيانات",
    title_en="LITTLE MCAR · MISSING DATA MECHANISMS",
    font_size=22,
)

<div dir="rtl" style="text-align:right">
## 4. كشف ارتباط غياب القيم بالمتغيرات المرصودة

تُصحح المقارنات بأسلوب Holm وتُسجل أحجام العينات المستخدمة لكل اختبار، وهو فحص استكشافي وليس برهانًا سببيًا.
</div>

In [ ]:
screen=kd.missingness_association_screen(dataset)
kd.display_analysis_dashboard(
    screen,theme="ocean",
    title_ar="تحليل ارتباط الفقد بالمؤشرات المرصودة",
    title_en="MISSINGNESS SCREEN · HOLM ADJUSTMENT",
    font_size=21,
)

<div dir="rtl" style="text-align:right">
## 5. التعويض المتعدد دون تسرب من الاختبار

تُدرَّب نماذج التعويض على أول 170 مشاهدة فقط، ثم تُطبق على الخمسين الباقية دون تدريب عليها. هذا فصل تعليمي لا يمثل بالضرورة أفضل تصميم زمني أو مجموعاتي لدراسة واقعية.
</div>

In [ ]:
train=dataset.iloc[:170].copy()
test=dataset.iloc[170:].copy()
result=kd.multiple_impute_train_test_numeric(
    train,test,n_imputations=5,seed=42,max_iter=10
)
print("Completed data versions:",result["n_imputations"])
display(result["imputations"][0][0].head())

<div dir="rtl" style="text-align:right">
## 6. دمج معامل انحدار بعد الاستكمالات

نُقدّر معامل إنفاق ← رضا بعد تعويض القيم المفقودة داخل مجموعة التدريب في كل مرة، ثم نجمع التقديرات وتبايناتها حسب Rubin. هذا مثال ارتباطي وليس سببيًا.
</div>

In [ ]:
import statsmodels.api as sm
estimates=[]
variances=[]
for completed_train,_ in result["imputations"]:
    fit=sm.OLS(
        completed_train["الرضا"],
        sm.add_constant(completed_train[["الإنفاق"]])
    ).fit()
    estimates.append(float(fit.params["الإنفاق"]))
    variances.append(float(fit.bse["الإنفاق"]**2))

rubin=kd.rubin_pool_scalar(estimates,variances)
kd.display_analysis_dashboard(
    rubin,theme="amethyst",
    title_ar="التقدير المجمّع وقواعد Rubin",
    title_en="MULTIPLE IMPUTATION · POOLED INFERENCE",
    font_size=22,
)

<div dir="rtl" style="text-align:right">
## 7. حفظ التقرير العلمي بتصميم مميز

يُنتج ملف HTML مستقلًا يدعم محاذاة العربي RTL والإنجليزي LTR، ويمكن طباعته إلى PDF من المتصفح.
</div>

In [ ]:
path=kd.save_analysis_dashboard(
    [little,screen,rubin],
    "KhaledDSML_Missing_Data_Academic_AR_EN.html",
    title_ar="التقرير الأكاديمي لتحليل البيانات المفقودة",
    title_en="ACADEMIC MISSING-DATA INFERENCE",
    theme="aurora",font_size=22
)
print(path)

<div dir="rtl" style="direction:rtl;text-align:right;font:650 16px Tahoma,Arial,sans-serif;line-height:1.9">
### متطلبات النشر العلمي

لا تعني عدم دلالة Little إثبات MCAR؛ ولا يثبت التعويض المتعدد غياب التحيز. دراسة MNAR تحتاج تحليلات حساسية ومعلومات إضافية. يجب مراجعة طبيعة المتغيرات وحدودها، والفصل العلمي بين التدريب والاختبار، واستقلال الوحدات، وملاءمة نموذج التعويض مع نموذج التحليل.
</div>